In [175]:
import random
import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

In [176]:
#Loading the new dataset with annotated reproducibility scores
file_path = "New Annotated Dataset.xlsx" 
sheet_index = 2  #Third sheet from the Excel file

df = pd.read_excel(file_path, sheet_name=sheet_index)

In [177]:
#Variable definitions for easier computation
factors = ['Data availability', 'Inclusion of a ReadMe', 'Trained model inclusion', 'Hyperparameter description', 'Training description',
    'Paper readability', 'Code availability'] #Reproducibility criteria

annotated_col = 'The reproducibility factor from the dataset by Olszewski et al.' #The annotated reproducibility factor

Evaluation of the Created Quantitative Reproducibility Measure for Machine Learning Research Papers:

Normalization of reproducibility criteria scores:

In [180]:
#Normalizing the reproducibility criteria scores with min-max
df_norm = df.copy() #Copy to not change the original data

#Looping through the criteria
for col in factors:
    min_val = df_norm[col].min() #Minimum value
    max_val = df_norm[col].max() #Maximum value
    df_norm[col] = (df_norm[col] - min_val) / (max_val - min_val) #Normalization

Calculating the reproducibility score with WSM:

In [182]:
#Calculating the reproducibility score with WSM. Without the weight selection for the reproducibility criteria, the calculation
#is just a mean.

df_norm['WSM_score'] = df_norm[factors].mean(axis=1)

In [183]:
k = len(factors)
weights = np.ones(k) / k

topsis_scores = []

for i in df_norm.index:
    scores = df_norm.loc[i, factors].values.astype(float)
    v = scores * weights
    d_best = np.sqrt(((v - weights) ** 2).sum())
    d_worst = np.sqrt((v ** 2).sum())
    topsis_scores.append(d_worst / (d_best + d_worst))

df_norm["TOPSIS_score"] = topsis_scores

Applying a threshold for a research paper to be classified as 0 or 1. Because the derived reproducibility scores with WSM were in the interval [0,1], a 0.5 threshold was selected to classify a paper as reproducible (value = 1). This was necessary as the reproducibility factor, from the dataset by D. Olszewski, et al., was provided as a binary variable containing only values of 0 and 1.

In [185]:
#Creating a column for the 0 and 1 labels derived from the WSM reproducibility calculation
df_norm['WSM_reproducible'] = (df_norm['WSM_score'] > 0.5).astype(int) 
df_norm['TOPSIS_reproducible'] = (df_norm['TOPSIS_score'] > 0.5).astype(int) 

#the 0.5 was chosen from a manual sensitivity analysis by increasing the threshold levels from 0 to 1, by 0.1 increments. 
#0.5 provided the most optimal solution with the highest level of accuracy, precision and recall.

Comparing the reproducibility score with WSM and the reproducibility factor, from the dataset by D. Olszewski, et al.:

In [187]:
#Copying the annotated reproducibility column in a dataframe, for easier calculation later
df_norm['Annotated_reproducible'] = df_norm[annotated_col]

#Creating a new column 'Match' that checks if WSM prediction matches the annotated reproducibility factor, for easier calculation later
df_norm['Match WSM'] = (df_norm['WSM_reproducible'] == df_norm['Annotated_reproducible'])
df_norm['Match TOPSIS'] = (df_norm['TOPSIS_reproducible'] == df_norm['Annotated_reproducible'])

In [188]:
#Calculating total papers, correctly predicted number of papers and accuracy

total_papers = len(df_norm) #Total number of papers
correct = df_norm['Match WSM'].sum() #Sum of paper with matching results
accuracy = correct / total_papers

#Displaying results
print("Total papers:", total_papers)
print("Correct predictions:", correct)
print("Accuracy:", round(accuracy, 4))

Total papers: 139
Correct predictions: 89
Accuracy: 0.6403


In [189]:
#Calculating total papers, correctly predicted number of papers and accuracy

total_papers = len(df_norm) #Total number of papers
correct = df_norm['Match TOPSIS'].sum() #Sum of paper with matching results
accuracy = correct / total_papers

#Displaying results
print("Total papers:", total_papers)
print("Correct predictions:", correct)
print("Accuracy:", round(accuracy, 4))

Total papers: 139
Correct predictions: 89
Accuracy: 0.6403


In [190]:
#Creating a confusion matrix to see results per reproducible and not reproducible paper type
confusion_matrix = pd.crosstab(
    df_norm['Annotated_reproducible'],
    df_norm['WSM_reproducible'],
    rownames=['Annotated'],
    colnames=['WSM']
)
#Displaying results
print("\nConfusion Matrix:")
print(confusion_matrix)


Confusion Matrix:
WSM         0   1
Annotated        
0          63  39
1          11  26


In [191]:
#Creating a confusion matrix to see results per reproducible and not reproducible paper type
confusion_matrix = pd.crosstab(
    df_norm['Annotated_reproducible'],
    df_norm['TOPSIS_reproducible'],
    rownames=['Annotated'],
    colnames=['TOPSIS']
)
#Displaying results
print("\nConfusion Matrix:")
print(confusion_matrix)


Confusion Matrix:
TOPSIS      0   1
Annotated        
0          63  39
1          11  26


In [192]:
#Calculating precision, recall and f-1
y_true = df_norm['Annotated_reproducible'] #True value labels from the reproducibility factor
y_pred = df_norm['WSM_reproducible'] #The labels derived from the WSM reproducibility calculation 

precision = precision_score(y_true, y_pred) #precision
recall = recall_score(y_true, y_pred) #recall
f1 = f1_score(y_true, y_pred) #f-1

#Displaying results
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1, 4))

Precision: 0.4
Recall: 0.7027
F1-score: 0.5098


In [193]:
#Calculating precision, recall and f-1
y_true = df_norm['Annotated_reproducible'] #True value labels from the reproducibility factor
y_pred = df_norm['TOPSIS_reproducible'] #The labels derived from the WSM reproducibility calculation 

precision = precision_score(y_true, y_pred) #precision
recall = recall_score(y_true, y_pred) #recall
f1 = f1_score(y_true, y_pred) #f-1

#Displaying results
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1, 4))

Precision: 0.4
Recall: 0.7027
F1-score: 0.5098


In [194]:
df_norm

,Paper number,Paper Title,Author,Link to Paper,Year,Data availability,Inclusion of a ReadMe,Trained model inclusion,Hyperparameter description,Training description,Paper readability,Code availability,The reproducibility factor from the dataset by Olszewski et al.,WSM_score,TOPSIS_score,WSM_reproducible,TOPSIS_reproducible,Annotated_reproducible,Match WSM,Match TOPSIS
0,704,DataLens: Scalable Privacy Preserving Training...,"Boxin Wang, Fan Wu, Yunhui Long, Luka Rimanic,...",https://dl.acm.org/doi/pdf/10.1145/3460120.348...,2021,1.000000,1.0,0.0,1.0,1.0,1.0,1.0,1,0.857143,0.710102,1,1,1,True,True
1,735,CRYPTGPU: Fast Privacy-Preserving Machine Lear...,"Sijun Tan, Brian Knott, Yuan Tian, David J. Wu",https://ieeexplore.ieee.org/stamp/stamp.jsp?tp...,2021,1.000000,0.5,0.0,0.5,0.5,1.0,1.0,0,0.642857,0.594131,1,1,0,False,False
2,546,SkillDetective: Automated Policy-Violation Det...,"Jeffrey Young, Song Liao, and Long Cheng, Hong...",https://www.usenix.org/system/files/sec22-youn...,2022,1.000000,1.0,0.0,0.5,0.5,0.5,1.0,0,0.642857,0.594131,1,1,0,False,False
3,634,Cost-Aware Robust Tree Ensembles for Security ...,"Yizheng Chen, Shiqi Wang, Weifan Jiang, Asaf C...",https://www.usenix.org/system/files/sec21-chen...,2021,0.666667,0.5,0.0,1.0,1.0,1.0,1.0,1,0.738095,0.650000,1,1,1,True,True
4,69,Latent Backdoor Attacks on Deep Neural Networks,"Yuanshun Yao, Huiying Li, \nHaitao Zheng, Ben ...",https://dl.acm.org/doi/pdf/10.1145/3319535.335...,2019,0.666667,0.0,0.0,1.0,0.5,1.0,0.0,1,0.452381,0.472393,0,0,1,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,480,Phishing URL Detection: A Network-based Approa...,"Taeri Kim, Noseong Park, Jiwon Hong, Sang-Wook...",https://doi.org/10.1145/3548606.3560615,2022,0.666667,0.5,0.0,1.0,1.0,1.0,1.0,0,0.738095,0.650000,1,1,0,False,False
135,92,Stealthy Adversarial Perturbations Against_x00...,"Shasha Li_x000D_\n, Ajaya Neupane_x000D_\n, Su...",https://www.cs.ucr.edu/~csong/ndss19-video.pdf,2019,1.000000,0.5,0.0,1.0,1.0,1.0,1.0,0,0.785714,0.672066,1,1,0,False,False
136,598,CV-INSPECTOR: Towards Automating Detection of ...,"Hieu Le, Athina Markopoulou, Zubair Shafiq",https://www.ndss-symposium.org/wp-content/uplo...,2021,0.666667,0.0,0.0,0.5,1.0,1.0,0.0,1,0.452381,0.472393,0,0,1,False,False
137,681,Dissecting Click Fraud Autonomy in the Wild,"Tong Zhu, Yan Meng, Haotian Hu, Xiaokuan Zhang...",https://dl.acm.org/doi/pdf/10.1145/3460120.348...,2021,0.333333,0.0,0.0,1.0,1.0,1.0,0.0,0,0.476190,0.487280,0,0,0,True,True


Benchmarking Results Against One/Two Criteria-Based Baselines:

In [196]:
#Creating binary variables for code and data availability (threshold = 0.5, as previuosly)
df_norm['Code_bin'] = (df_norm['Code availability'] >= 0.5).astype(int)
df_norm['Data_bin'] = (df_norm['Data availability'] >= 0.5).astype(int)

#Ground truth labels (the reproducibility factor, from the dataset by D. Olszewski, et al.)
y_true = df_norm[annotated_col]

#Predictions for each baseline scenario
y_pred_code = df_norm['Code_bin']
y_pred_data = df_norm['Data_bin']
y_pred_or = ((df_norm['Code_bin'] + df_norm['Data_bin']) >= 1).astype(int)
y_pred_and = ((df_norm['Code_bin'] + df_norm['Data_bin']) == 2).astype(int)

In [197]:
#Main function to compute evaluation metrics for the baseline models
def evaluate_model(name, y_true, y_pred):
 return {
 'Model': name,
 'Precision': precision_score(y_true, y_pred),
 'Recall': recall_score(y_true, y_pred),
 'F1-score': f1_score(y_true, y_pred)
}

In [198]:
#Storing results and evaluating all baseline models
results = []
results.append(evaluate_model("Code availability only", y_true, y_pred_code))
results.append(evaluate_model("Data availability only", y_true, y_pred_data))
results.append(evaluate_model("Code or Data", y_true, y_pred_or))
results.append(evaluate_model("Code and Data", y_true, y_pred_and))

In [199]:
#Printing and sorting by f1-score to compare performance
benchmark_df = pd.DataFrame(results)
print(benchmark_df.sort_values(by="F1-score", ascending=False))

                    Model  Precision    Recall  F1-score
1  Data availability only   0.329268  0.729730  0.453782
2            Code or Data   0.301887  0.864865  0.447552
0  Code availability only   0.273810  0.621622  0.380165
3           Code and Data   0.300000  0.486486  0.371134


Example Analysis of the Quantitative Reproducibility Measure for Machine Learning Research Papers:

Criteria evaluation with the Saaty's importance scale for three different scenarios:

In [202]:
#Scenario importance score vectors for the three weight scenarios
scenario_scores = {
    "Scenario_1_equal": np.array([1, 1, 1, 1, 1, 1, 1], dtype=float), #Equal weights scenario
    "Scenario_2_data_code": np.array([7, 3, 3, 3, 3, 3, 7], dtype=float), #Data and code focused scenario
    "Scenario_3_read_train": np.array([3, 3, 1, 4, 7, 7, 3], dtype=float), #Readability and training focused scenario
}

RI = 1.32  #Random Index for n=7 (n=7 because there are 7 criteria)

Functions to create a pairwise comparison matrix equivalent and for the consistency ratio evaluation for the weights:

In [204]:
#Converting a vector of scores into normalized weights that sum to 1

def scores_to_weights(scores): #Normalizing raw scores so that their sum equals 1
    return scores / scores.sum()

def build_pairwise_matrix(scores): #Building a pairwise comparison matrix from a score vector. Each element A[i, j] = scores[i] / scores[j]
    return scores[:, None] / scores[None, :]

#Get AHP priority weights from the pairwise comparison matrix
def ahp_weights(A):
    A_norm = A / A.sum(axis=0) #Normalization of each column of the matrix
    return A_norm.mean(axis=1) #Mean of each row to obtain the weight vector

#Consistency metrics for the AHP pairwise comparison matrix
def consistency_ratio(A, weights, RI): #Pairwise comparison matrix, AHP-derived weight vector, Random index for matrices of size n
    n = A.shape[0]
    lambda_max = np.mean((A @ weights) / weights) #Maximum eigenvalue
    CI = (lambda_max - n) / (n - 1) #Consistency index
    CR = CI / RI #Consistency ratio
    return lambda_max, CI, CR


Consistency index for the weights in weight scenarios:

In [206]:
#Creating a list for results for each scenario
results = []

#Loop through each scenario and it's score vector
for name, scores in scenario_scores.items():
    A = build_pairwise_matrix(scores) #Creating a AHP pairwise comparison matrix from the score vector
    w = ahp_weights(A) #AHP priority weights from the pairwise matrix
    lambda_max, CI, CR = consistency_ratio(A, w, RI) #Consistency metrics
    results.append([name, lambda_max, CI, CR])

#Converting results into a dataframe for easier results vizualization
cr_df = pd.DataFrame(
    results,
    columns=["Scenario", "lambda_max", "CI", "CR"]
)
#Displaying results
print(cr_df)

                Scenario  lambda_max   CI   CR
0       Scenario_1_equal         7.0  0.0  0.0
1   Scenario_2_data_code         7.0  0.0  0.0
2  Scenario_3_read_train         7.0  0.0  0.0


Normalizing weights for each weight scenario:

In [208]:
#Creating normalized weights for each scenario

weights = {name: scores_to_weights(s) for name, s in scenario_scores.items()} #Each score vector is converted into weights that sum to 1
weights_df = pd.DataFrame(weights, index=factors) #Converting the weights dictionary into a dataframe for easier application in next parts of the code

#Displaying results
print("\nScenario weights:")
print(weights_df)
print("\nColumn sums:")
print(weights_df.sum(axis=0))


Scenario weights:
                            Scenario_1_equal  Scenario_2_data_code  \
Data availability                   0.142857              0.241379   
Inclusion of a ReadMe               0.142857              0.103448   
Trained model inclusion             0.142857              0.103448   
Hyperparameter description          0.142857              0.103448   
Training description                0.142857              0.103448   
Paper readability                   0.142857              0.103448   
Code availability                   0.142857              0.241379   

                            Scenario_3_read_train  
Data availability                        0.107143  
Inclusion of a ReadMe                    0.107143  
Trained model inclusion                  0.035714  
Hyperparameter description               0.142857  
Training description                     0.250000  
Paper readability                        0.250000  
Code availability                        0.107143  

Col

Computing WSM scores for different scenarios:

In [210]:
#Looping through each scenario and its corresponding weight vector
for name, w in weights.items():
    df_norm[f"WSM_{name}"] = df_norm[factors].values @ w 
    
#Computing the WSM score for each row: WSM = sum of (criteria value × corresponding weight). The @ operator performs matrix multiplication

Computing TOPSIS scores for different scenarios:

In [212]:
#Decision matrix (same as WSM)
D = df_norm[factors].values.astype(float)
norm = np.sqrt((D ** 2).sum(axis=0))
R = D / norm

#TOPSIS for each scenario
for name, w in weights.items():
    V = R * w
    #Ideal best and worst (all criteria are benefit criteria)
    ideal_best = np.max(V, axis=0)
    ideal_worst = np.min(V, axis=0)
    #Distance to ideal best and worst
    dist_best = np.sqrt(((V - ideal_best) ** 2).sum(axis=1))
    dist_worst = np.sqrt(((V - ideal_worst) ** 2).sum(axis=1))
    #Closeness coefficient
    closeness = dist_worst / (dist_best + dist_worst)
    #Store results
    df_norm[f"TOPSIS_{name}"] = closeness

Randomly sample 10 papers from the new annotated dataset:

In [214]:
#Randomly sample 10 papers from the new annotated dataset. it was chosen to sample 10 research papers that were reproducible, 
#according to the reproducibility factor from D. Olszewski, et al. \cite{olszewski2023get}, because this subset allows for observing 
#a wider range of variability in reproducibility scores under different weight assignment procedure specifications.

sample_df = (df_norm[df_norm[annotated_col] == 1].sample(n=10, random_state=42).reset_index(drop=True))

#Save as a csv file
sample_df.to_csv("10_annotated_papers.csv", index=False)

#Display results
cols = ["Paper number", "Paper Title"] + [f"WSM_{k}" for k in weights] + [f"TOPSIS_{k}" for k in weights] #WSM score for each weight scenario with Paper number and title
print(sample_df[cols])

   Paper number                                        Paper Title  \
0           271  WTAGRAPH: Web Tracking and Advertising Detecti...   
1           461  DeepSteal: Advanced Model Extractions Leveragi...   
2           269  Model Stealing Attacks Against Inductive Graph...   
3           613  Phishpedia: A Hybrid Deep Learning Based Appro...   
4           552  Cheetah: Lean and Fast Secure Two-Party Deep N...   
5           459  DEEPCASE: Semi-Supervised Contextual Analysis ...   
6           556  Khaleesi: Breaker of Advertising and Tracking ...   
7           257  AI/ML for Network Security: The Emperor has no...   
8           740  Improving Password Guessing via Representation...   
9            80  Privacy Risks of Securing Machine Learning Mod...   

   WSM_Scenario_1_equal  WSM_Scenario_2_data_code  WSM_Scenario_3_read_train  \
0              0.285714                  0.206897                   0.446429   
1              0.428571                  0.448276                   0

Weight sensitivity analysis for TOPSIS and WSM for the 3 AHP weight scenarios:

In [216]:
#Extracting only the selected sample of papers
X_sample = sample_df[factors].values

#Normalizing matrix for TOPSIS (vector normalization per criterion)
norm = np.sqrt((X_sample ** 2).sum(axis=0))
R_sample = X_sample / norm

In [217]:
#Weight perturbation function
def perturb_weights(w, perturbation=0.2): #random ±20% noise to each weight
    noise = np.random.uniform(-perturbation, perturbation, size=len(w))
    w_new = w * (1 + noise) #renormalize the vector
    w_new = np.clip(w_new, 1e-8, None)
    return w_new / w_new.sum()

In [218]:
#Monte Carlo sensitivity analysis function. The weight uncertainty is simulated by perturbing AHP weights (±20%) and recomputing WSM and TOPSIS scores.
n_sim = 500 #500 simulations
sensitivity_results = {}

for name, w in weights.items():
    wsm_scores_all = []
    topsis_scores_all = []
    for _ in range(n_sim):
        w_perturbed = perturb_weights(w) #Perturbing weights (±20%) and renormalizing
        wsm_scores = X_sample @ w_perturbed #WSM computation
        V = R_sample * w_perturbed #TOPSIS computation:
        ideal_best = np.max(V, axis=0)
        ideal_worst = np.min(V, axis=0)
        dist_best = np.sqrt(((V - ideal_best) ** 2).sum(axis=1))
        dist_worst = np.sqrt(((V - ideal_worst) ** 2).sum(axis=1))
        topsis_scores = dist_worst / (dist_best + dist_worst)
        wsm_scores_all.append(wsm_scores) #Store results for WSM
        topsis_scores_all.append(topsis_scores) #Store results for TOPSIS
    
    sensitivity_results[name] = {
        "wsm": np.array(wsm_scores_all),
        "topsis": np.array(topsis_scores_all)
    }

In [219]:
#Function for descriptive statistics for score distributions (mean, min, max and percentiles)
def summarize(scores):
    return {
        "mean": scores.mean(axis=0),
        "min": scores.min(axis=0),
        "max": scores.max(axis=0),
        "p5": np.percentile(scores, 5, axis=0),
        "p95": np.percentile(scores, 95, axis=0)
    }

In [220]:
#WSM sensitivity results with descriptive statistics
print("\nWSM Sensitivity:")
for name in sensitivity_results: 
    print(f"\n{name}")
    wsm_stats = summarize(sensitivity_results[name]["wsm"])
    df_wsm = pd.DataFrame({
        "Paper": sample_df["Paper number"],
        "Mean": wsm_stats["mean"],
        "Min": wsm_stats["min"],
        "Max": wsm_stats["max"],
        "P5": wsm_stats["p5"],
        "P95": wsm_stats["p95"]
    })
    display(df_wsm)


WSM Sensitivity:

Scenario_1_equal


,Paper,Mean,Min,Max,P5,P95
0,271,0.285562,0.234940,0.331005,0.260159,0.312950
1,461,0.428127,0.377767,0.474398,0.399982,0.455383
2,269,0.452985,0.398556,0.500087,0.422649,0.482896
3,613,0.642577,0.602630,0.689063,0.617513,0.667015
4,552,0.522422,0.476314,0.574383,0.494702,0.549659
5,459,0.642675,0.602123,0.681349,0.615342,0.668323
6,556,0.594372,0.561095,0.628333,0.573005,0.617635
7,257,0.785222,0.752993,0.823644,0.766637,0.803058
8,740,0.523769,0.457140,0.582288,0.493693,0.558262
9,80,0.596944,0.542608,0.643906,0.566362,0.625289



Scenario_2_data_code


,Paper,Mean,Min,Max,P5,P95
0,271,0.206666,0.173091,0.244500,0.183564,0.229035
1,461,0.448973,0.394193,0.501811,0.409783,0.488372
2,269,0.420204,0.372413,0.469772,0.389710,0.450547
3,613,0.604398,0.550525,0.661080,0.566822,0.637002
4,552,0.608431,0.560682,0.648512,0.583967,0.635582
5,459,0.741412,0.693778,0.781352,0.715486,0.765971
6,556,0.591884,0.559009,0.622525,0.574174,0.609303
7,257,0.844823,0.814582,0.869344,0.828414,0.861102
8,740,0.471693,0.416120,0.527712,0.438958,0.505764
9,80,0.660898,0.614636,0.701072,0.635657,0.685324



Scenario_3_read_train


,Paper,Mean,Min,Max,P5,P95
0,271,0.446535,0.397325,0.496398,0.414008,0.476740
1,461,0.499865,0.466985,0.542830,0.480037,0.522517
2,269,0.588806,0.539507,0.634255,0.562014,0.615886
3,613,0.643208,0.603733,0.677058,0.616633,0.665535
4,552,0.554079,0.505418,0.614305,0.519737,0.590510
5,459,0.714424,0.676927,0.743250,0.693468,0.734002
6,556,0.750367,0.719310,0.787513,0.729083,0.771076
7,257,0.678853,0.647085,0.708511,0.661072,0.696802
8,740,0.714194,0.665089,0.766033,0.684740,0.744876
9,80,0.695796,0.654581,0.734206,0.670953,0.719196


In [221]:
#TOPSIS sensitivity results with descriptive statistics
print("\nTOPSIS Sensitivity:")
for name in sensitivity_results: 
    print(f"\n{name}")
    topsis_stats = summarize(sensitivity_results[name]["topsis"])
    df_topsis = pd.DataFrame({
        "Paper": sample_df["Paper number"],
        "Mean": topsis_stats["mean"],
        "Min": topsis_stats["min"],
        "Max": topsis_stats["max"],
        "P5": topsis_stats["p5"],
        "P95": topsis_stats["p95"]
    })
    display(df_topsis)


TOPSIS Sensitivity:

Scenario_1_equal


,Paper,Mean,Min,Max,P5,P95
0,271,0.190903,0.148679,0.229472,0.166958,0.216385
1,461,0.299507,0.250125,0.355095,0.266295,0.334062
2,269,0.311928,0.262367,0.360527,0.280469,0.345982
3,613,0.589944,0.537811,0.652815,0.551754,0.626902
4,552,0.429273,0.366511,0.498240,0.390979,0.470898
5,459,0.491576,0.425070,0.553610,0.449520,0.533271
6,556,0.436038,0.374618,0.485913,0.400786,0.470850
7,257,0.769250,0.719014,0.822038,0.743555,0.795438
8,740,0.332904,0.281252,0.383154,0.302282,0.366541
9,80,0.409129,0.337816,0.479766,0.367569,0.448518



Scenario_2_data_code


,Paper,Mean,Min,Max,P5,P95
0,271,0.137505,0.109154,0.178346,0.117786,0.156879
1,461,0.389105,0.319416,0.459371,0.337333,0.439608
2,269,0.329967,0.275231,0.388675,0.289751,0.370365
3,613,0.514229,0.438349,0.589836,0.460704,0.560091
4,552,0.585053,0.523992,0.644486,0.544787,0.626138
5,459,0.648124,0.580627,0.706462,0.605802,0.686684
6,556,0.493205,0.449292,0.528946,0.465415,0.519187
7,257,0.833063,0.794595,0.865234,0.812075,0.854603
8,740,0.340490,0.283614,0.401317,0.300771,0.380531
9,80,0.560323,0.484991,0.624401,0.519567,0.603588



Scenario_3_read_train


,Paper,Mean,Min,Max,P5,P95
0,271,0.376405,0.322660,0.440107,0.340417,0.408920
1,461,0.374181,0.320103,0.417840,0.345373,0.403240
2,269,0.486583,0.418106,0.554556,0.438604,0.530938
3,613,0.562871,0.501792,0.615405,0.518997,0.599864
4,552,0.443187,0.389154,0.513999,0.404298,0.486168
5,459,0.617135,0.551787,0.666450,0.579666,0.653012
6,556,0.663053,0.617776,0.719298,0.629878,0.695343
7,257,0.595994,0.537653,0.653472,0.563821,0.629306
8,740,0.550009,0.480794,0.629310,0.503237,0.595528
9,80,0.546211,0.466459,0.610387,0.501026,0.590913


In [222]:
#Function to compute average score range across simulations for each scenario
for name in sensitivity_results:
    wsm = sensitivity_results[name]["wsm"]
    topsis = sensitivity_results[name]["topsis"]
    wsm_range = (wsm.max(axis=0) - wsm.min(axis=0)).mean() #Basic average formula
    topsis_range = (topsis.max(axis=0) - topsis.min(axis=0)).mean()
    
    print(f"{name}:")
    print(f"WSM avg variation ≈ {wsm_range:.3f}")
    print(f"TOPSIS avg variation ≈ {topsis_range:.3f}")

Scenario_1_equal:
WSM avg variation ≈ 0.092
TOPSIS avg variation ≈ 0.112
Scenario_2_data_code:
WSM avg variation ≈ 0.088
TOPSIS avg variation ≈ 0.113
Scenario_3_read_train:
WSM avg variation ≈ 0.083
TOPSIS avg variation ≈ 0.121
